In [18]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer  
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

import joblib
import os

In [13]:
#Columns Groups

ord_cols = [
    "sub_grade", "emp_length"
]

ord_cats = [
    ["A1","A2","A3","A4","A5", 
     "B1","B2","B3","B4","B5",
     "C1","C2","C3","C4","C5", 
     "D1","D2","D3","D4","D5",
     "E1","E2","E3","E4","E5", 
     "F1","F2","F3","F4","F5",
     "G1","G2","G3","G4","G5"],

    ["Missing", "< 1 year",
     "1 year", "2 years",
     "3 years", "4 years",
     "5 years", "6 years",
     "7 years", "8 years",
     "9 years", "10+ years"]
]

cat_cols = [
    "home_ownership", "verification_status",
    "purpose", "zip_code", "addr_state",
    "initial_list_status", "application_type",
    "disbursement_method"
]

target_col = "default_flag"

num_structural = [
    "mths_since_last_delinq", 
    "mths_since_last_record",  
    "mths_since_last_major_derog",  
    "mths_since_recent_bc_dlq",
    "mths_since_recent_revol_delinq",  
    "mths_since_rcnt_il",
    "mths_since_recent_bc",
    "mths_since_recent_inq",
]

exclude = cat_cols + ord_cols + [target_col] + num_structural
num_rest = [col for col in train_df.columns if col not in exclude]

print("Numeric (rest):", len(num_rest))
print("Ordinal:", len(ord_cols))
print("Categorical:", len(cat_cols))


Numeric (rest): 79
Ordinal: 2
Categorical: 8


In [16]:
#Initialize Preprocessor (Non XGBoost)

iter_imputer = IterativeImputer(
    estimator=RandomForestRegressor(
        n_estimators=10,   
        random_state=42
    ),
    max_iter=5,
    initial_strategy="median",
    skip_complete=True,
    random_state=42
)

num_rest_transformer = Pipeline(steps=[
    ("imputer1", iter_imputer),
    ('scaler', StandardScaler())])

num_struct_transformer = Pipeline(steps=[
    ("imputer2", SimpleImputer(strategy="constant", fill_value=999)),
    ('scaler2', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

ordinal_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories = ord_cats))])

sk_preprocessor = ColumnTransformer(
    transformers=[
        ('num1', num_rest_transformer, num_rest),
        ('num2', num_struct_transformer, num_structural),
        ('cat', categorical_transformer, cat_cols),
        ('ord', ordinal_transformer, ord_cols)])

In [17]:
num_cols = num_rest + num_structural

xg_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("ord", OrdinalEncoder(categories=ord_cats), ord_cols),
    ]
)

In [20]:
dir = "../data/modeling"

# File paths
sk_path = os.path.join(dir, "sk_preprocessor.pkl")
xg_path = os.path.join(dir, "xg_preprocessor.pkl")

# Save preprocessors
joblib.dump(sk_preprocessor, sk_path)
joblib.dump(xg_preprocessor, xg_path)

print("Saved sklearn preprocessor →", sk_path)
print("Saved XGBoost preprocessor →", xg_path)

Saved sklearn preprocessor → ../data/modeling/sk_preprocessor.pkl
Saved XGBoost preprocessor → ../data/modeling/xg_preprocessor.pkl
